In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv('/content/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [16]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [17]:
x = df.iloc[:,0:-1].values
y = df.iloc[:,-1].values

In [18]:
x

array([[  6.   , 148.   ,  72.   , ...,  33.6  ,   0.627,  50.   ],
       [  1.   ,  85.   ,  66.   , ...,  26.6  ,   0.351,  31.   ],
       [  8.   , 183.   ,  64.   , ...,  23.3  ,   0.672,  32.   ],
       ...,
       [  5.   , 121.   ,  72.   , ...,  26.2  ,   0.245,  30.   ],
       [  1.   , 126.   ,  60.   , ...,  30.1  ,   0.349,  47.   ],
       [  1.   ,  93.   ,  70.   , ...,  30.4  ,   0.315,  23.   ]])

In [19]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [20]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2, random_state = 42)

In [22]:
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [23]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [24]:
model = Sequential()
model.add(Dense(32, activation = 'relu', input_dim = 8))
model.add(Dense(1, activation = 'sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics = ['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
model.fit(x_train,y_train, batch_size = 32, epochs = 100, validation_data = (x_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6352 - loss: 0.7256 - val_accuracy: 0.6429 - val_loss: 18.2764
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.6578 - val_accuracy: 0.6039 - val_loss: 8.9768
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6824 - loss: 0.6092 - val_accuracy: 0.3636 - val_loss: 10.6052
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7166 - loss: 0.5707 - val_accuracy: 0.3442 - val_loss: 17.9743
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7557 - loss: 0.5419 - val_accuracy: 0.3506 - val_loss: 26.4154
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.5203 - val_accuracy: 0.3571 - val_loss: 34.0150
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7622 - loss: 0.5036 - val_accuracy: 0.3571 - val_loss: 39.4844
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7655 - loss: 0.4916 - val_accuracy: 0.3

In [26]:
# 1. How to select appropriate Optimizer
# 2. No. of Nodes in a layer
# 3. How to select no. of Layers.
# 4. All in one Model

In [28]:
pip install -U keras-tuner

In [29]:
import kerastuner as kt

/tmp/ipykernel_882/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [30]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32, activation ='relu', input_dim = 8))
  model.add(Dense(1, activation = 'sigmoid'))

  optimizer = hp.Choice('optimizer', values = ['adam','sgd','rmsprop','adadelta'])

  model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

  return model

In [34]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 5)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [35]:
tuner.search(x_train,y_train, epochs = 5, validation_data =(x_test,y_test))

Trial 4 Complete [00h 00m 04s]
val_accuracy: 0.5779221057891846

Best val_accuracy So Far: 0.6753246784210205
Total elapsed time: 00h 00m 22s


In [39]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [40]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [42]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
model.fit(x_train,y_train,batch_size=32,epochs = 100, initial_epoch = 6, validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.8127 - loss: 0.4004 - val_accuracy: 0.3571 - val_loss: 84.7896
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8046 - loss: 0.4006 - val_accuracy: 0.3571 - val_loss: 85.8788
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8094 - loss: 0.4000 - val_accuracy: 0.3571 - val_loss: 83.7980
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8111 - loss: 0.3996 - val_accuracy: 0.3571 - val_loss: 84.0196
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8094 - loss: 0.3993 - val_accuracy: 0.3571 - val_loss: 83.9834
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8111 - loss: 0.3984 - val_accuracy: 0.3571 - val_loss: 84.1166
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8094 - loss: 0.3984 - val_accuracy: 0.3571 - val_loss: 85.7357
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8143 - loss: 0.3979 - val_a

In [53]:
def build_model(hp):

  model = Sequential()

  units = hp.Int('units' ,min_value = 8, max_value = 128 ,step = 8)

  model.add(Dense(units = units, activation = 'relu', input_dim = 8))
  model.add(Dense(1,activation = 'sigmoid'))

  model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

  return model

In [54]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 5,
                        directory = 'mydir',
                        project_name = 'first')

In [55]:
tuner.search(x_train,y_train,epochs = 5,validation_data =(x_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.3571428656578064

Best val_accuracy So Far: 0.7077922224998474
Total elapsed time: 00h 00m 16s


In [57]:
tuner.get_best_hyperparameters()[0].values

{'units': 32}

In [59]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [60]:
model.fit(x_train,y_train,batch_size = 32, epochs = 100, initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.3404 - loss: 0.7801
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4446 - loss: 0.7088
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6450 - loss: 0.6583
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7199 - loss: 0.6214
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7427 - loss: 0.5919
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.5693
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.5505
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.5359
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7720 - loss: 0.5234
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7720 - loss: 0.5129
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.5026
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc

In [77]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32,input_dim = 8,activation = 'relu'))

  for i in range(hp.Int('num_layers', min_value = 1, max_value = 10)):

    model.add(Dense(72,activation = 'relu'))

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

  return model



In [78]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 3,
                        directory = 'mydir',
                        project_name = 'second')

Reloading Tuner from mydir/second/tuner0.json


In [79]:
tuner.search(x_train,y_train,epochs = 5, validation_data = (x_test,y_test))

In [80]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 10}

In [81]:
model = tuner.get_best_models(num_models = 1)[0]

In [82]:
model.fit(x_train,y_train,epochs = 20,initial_epoch = 6,validation_data =(x_test,y_test))

Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.6645 - loss: 0.5545 - val_accuracy: 0.3571 - val_loss: 0.7513
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7687 - loss: 0.4987 - val_accuracy: 0.3571 - val_loss: 4.3747
Epoch 9/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7818 - loss: 0.4547 - val_accuracy: 0.3377 - val_loss: 19.4174
Epoch 10/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7785 - loss: 0.4567 - val_accuracy: 0.3571 - val_loss: 22.1035
Epoch 11/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7948 - loss: 0.4254 - val_accuracy: 0.3571 - val_loss: 21.5370
Epoch 12/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8078 - loss: 0.4081 - val_accuracy: 0.3571 - val_loss: 31.9280
Epoch 13/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8143 - loss: 0.4150 - val_accuracy: 0.3442 - val_loss: 26.3087
Epoch 14/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8274 - loss: 0.3849 - val_accuracy: 0.35